# 03 · RAG 基本架构：每一层为什么存在

> 大纲里的核心要求是：**理解每一层为什么存在，而不是只会调用 LangChain**。本课把 RAG 流水线逐层拆开。

**本文件覆盖知识点**：Document → Loader → Parsing → Chunking → Embedding → Vector DB → Retriever → Reranker → Context → LLM → Answer

In [ ]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


## 1. 标准流水线（每一层=一个问题）

```text
Documents(原材料)           为什么需要它
   │
Document Loader        文件格式五花八门，先读进来
   │
Document Parsing       PDF 里还有表格/图片/版面，要结构化
   │
Chunking              太长编不成一个向量，切成语义块
   │
Embedding             文本->向量，才能算“语义距离”
   │
Vector Database       海量向量要有地方存、要能快速查
   │
Retriever             给定问题，找出最像的 Top-K
   │
Reranker              粗召回里再精排，把最贴的顶上来
   │
Context               拼成给模型的“参考资料”
   │
LLM                   基于资料生成回答
   │
Answer
```

本课程将按这套图的顺序逐层深讲。先把“为什么需要这一层”刻进脑子，再学每层的具体做法。

## 2. 两个阶段

- **离线索引（Indexing）**：上面的 Loader→Parsing→Chunking→Embedding→VectorDB，只在知识库更新时执行一次，允许慢、可批量；
- **在线查询（Querying）**：下面的 Retriever→Reranker→Context→LLM，**每次提问都要走一遍**，必须快。

理解这条时间线很重要：所有“慢活”都尽量挪到离线，在线只保留“能优化的快活”。

In [ ]:
# 用一个接口先行(minimal skeleton) 把整条流水线的“形状”立起来
# 后面的课逐个填实现；这里只让你看清 每层的位置与职责。
class RAGPipeline:
    """RAG 流水线骨架：每层一个方法，后续 notebook 逐个实现"""

    # ---- 离线阶段 ----
    def load(self, path): pass            # 04 文档加载
    def parse(self, doc): pass            # 05 结构化解析
    def clean(self, text): pass           # 06 数据清洗
    def chunk(self, text): pass           # 06 清洗 · 07 分块
    def embed(self, texts): pass          # 10 Embedding
    def build_index(self, vecs): pass     # 13 向量库

    # ---- 在线阶段 ----
    def retrieve(self, query, k): pass    # 15 检索
    def rerank(self, query, cands): pass  # 22 精排
    def make_context(self, docs): pass    # 23 上下文工程
    def generate(self, ctx, q): pass      # 25 提示词与引文

    def ask(self, q):
        docs = self.retrieve(q, k=10)
        docs = self.rerank(q, docs)
        ctx = self.make_context(docs)
        return self.generate(ctx, q)

print('骨架已立：ask() 就是"检索->精排->上下文->生成"。后续课程逐一填充。')

In [ ]:
# 知识点·真调说明：Retriever/Reranker/Context 为何存在 —— 同一问题，召回片段“沾边但无答案”vs“正中要害”，答案天差地别
print('① 模拟召回只命中关键词、没把最相关的片段顶上来（给的片段不含答案）')
_llm_live(
    prompt='【参考资料】企业把常见问题与产品文档导入知识库后，系统会对查询做向量相似度检索，'
           '常用的相似度度量有余弦相似度、欧氏距离与内积；写库前一般先做 L2 归一化。\n'
           '【问题】星云客服机器人如果答不上来，会自动转人工吗？会带上什么？',
    system='你是“星云智能”客服 AI。只依据【参考资料】回答，资料里没写的一律答“资料未提及”，不要自行脑补。',
    fallback='资料只讲了“向量相似度检索/度量”，并没有提“转人工”——严谨的模型只能答“资料未提及”；'
             '若硬答，只能拿通用印象编，无法保证与真实产品一致。',
    temperature=0.2,
)
print()
print('② 模拟更精准的召回/重排把“真命中文档”顶了上来（给的片段正含答案）')
_llm_live(
    prompt='【参考资料】管理员把企业的 FAQ 与产品文档导入知识库，机器人回答时会先检索知识库，'
           '再结合大模型生成回复；当机器人无法解决客户问题时，会自动转接人工客服，并携带完整的对话上下文。\n'
           '【问题】星云客服机器人如果答不上来，会自动转人工吗？会带上什么？',
    system='你是“星云智能”客服 AI。只依据【参考资料】回答，资料没写的一律答“资料未提及”。',
    fallback='依据资料可稳答：会。当机器人无法解决客户问题时自动转接人工客服，并携带完整的对话上下文。',
    temperature=0.2,
)
print()
print('同样走“检索→拼上下文→生成”，片段选得好不好，直接决定能否答出关键信息。')
print('→ 这正是 Retriever→Reranker→Context 要各自成层、并不断被优化的原因：'
      '“召回错位/混入噪声”是 RAG 答案变差的最大来源之一，本课先记住每一层都在解决这一类问题。')

## 3. 架构演进：Naive → Advanced → Modular

- **Naive RAG**：加载→切分→向量→检索→拼接→生成（本课后面主要实现这条）;
- **Advanced RAG**：在前后加“优化器”——查询改写、混合检索、重排、上下文压缩（第 17~24 课）;
- **Modular RAG**：把各层做成可插拔模块，可按需组合（Graph、SQL、Agentic…，第 27~32 课）。

> 学习顺序建议：先把 Naive 每条链路亲手跑通，再往 Advanced 加模块。



In [ ]:
# 知识点·真调说明：分层架构的价值 —— 让模型当“体检医生”，把故障现象归因到具体层
_llm_live(
    prompt='一个 RAG 问答 App 上线后收到三类故障，请把每个故障归因到'
           '“Loader / Parsing / Chunking / Embedding / VectorDB / Retriever / Reranker / Context / LLM”'
           '中的 1~2 层，并给一句该层该查什么：\n'
           '① 用户问“基础版和专业版差在哪”，回答里却混进了一段完全无关的《物流退货政策》；\n'
           '② 两份文档都讲“私有化部署”但指的是不同产品，模型把两个概念搅在一起答串了；\n'
           '③ 回答引用的页码是错的，用户翻开发现那一页根本没有这段内容。',
    system='你是 RAG 系统诊断专家。逐条按“故障 → 可能出错层(1~2个) → 一句话排查点”作答，每条不超过 2 行，直接给结论。',
    fallback='① → Retriever / Reranker：召回把词面近但不相关的片段也带进来了，先查是否缺重排、检索词是否太泛。\n'
             '② → Chunking / Context：两个同名概念没靠标题/元数据分开，模型无法区分；应在切分时保留标题、拼上下文时带上出处。\n'
             '③ → Parsing / 元数据：页码从解析阶段就绑错了，查解析环节页码与正文的绑定，别信后加的假页码。',
    temperature=0.2,
)
print('→ 一个故障能定位到某一层，正说明分层架构“可定位、可替换”：'
      '理解每一层为什么存在，出了故障才指得对地方——这就是本课“每层=一个问题”的用意。')

## 小结

- RAG 是一条 **十层流水线**，分**离线索引**与**在线查询**两阶段；
- 学每一层都问“它解决什么问题、不加它会怎样”；
- 从 Naive RAG 出发，逐步升级到 Advanced / Modular RAG。